# 手写礼簿识别自动化流水线

本 notebook 实现以下功能：
1. 批量读取 `images/` 文件夹中的礼簿图片
2. 使用 TextIn API 进行 OCR 识别
3. 使用 LLM (GPT) 进行智能纠错
4. 生成对应的 CSV 文件（支持断点恢复）
5. 合并所有 CSV 形成最终礼簿

In [10]:
# 导入必要的库
import os
import json
import glob
import pandas as pd
from pathlib import Path

# 导入 cn2an 库用于数字转中文大写金额
try:
    import cn2an
    print("cn2an 库已导入成功")
except ImportError:
    print("cn2an 库未安装，正在安装...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cn2an"])
    import cn2an
    print("cn2an 库安装成功")

# 配置路径
IMAGES_DIR = "images"  # 图片文件夹
OUTPUT_DIR = "output"  # 输出文件夹

# 确保输出目录存在
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"图片目录: {IMAGES_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

cn2an 库已导入成功
图片目录: images
输出目录: output


In [11]:
# 检查并清空 目录
if os.path.exists("images"):
    # 清空目录下所有文件
    for filename in os.listdir("images"):
        file_path = os.path.join("images", filename)
        if os.path.isfile(file_path):
            os.remove(file_path)
else:
    os.makedirs("images")

# 检查并清空 目录
if os.path.exists("output"):
    # 清空目录下所有文件
    for filename in os.listdir("output"):
        file_path = os.path.join("output", filename)
        if os.path.isfile(file_path):
            os.remove(file_path)
else:
    os.makedirs("output")

In [12]:
# 查看 images 文件夹中的图片并重命名为 001.png, 002.png, ...
image_files = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.png")))
print(f"共找到 {len(image_files)} 张图片:")
for idx, img in enumerate(image_files, 1):
    new_name = f"{idx:03d}.png"
    new_path = os.path.join(IMAGES_DIR, new_name)
    # 仅当原名不同于目标名才重命名，避免无意义覆盖
    if os.path.basename(img) != new_name:
        print(f"  - {os.path.basename(img)} -> {new_name}")
        os.rename(img, new_path)
    else:
        print(f"  - {new_name} (已命名)")
# 重命名后再获取排序后的新文件名列表
image_files = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.png")))

共找到 55 张图片:
  - iShot_2026-02-26_19.26.39.png -> 001.png
  - iShot_2026-02-26_19.29.35.png -> 002.png
  - iShot_2026-02-26_19.30.36.png -> 003.png
  - iShot_2026-02-26_19.30.49.png -> 004.png
  - iShot_2026-02-26_19.30.58.png -> 005.png
  - iShot_2026-02-26_19.31.04.png -> 006.png
  - iShot_2026-02-26_19.31.13.png -> 007.png
  - iShot_2026-02-26_19.31.24.png -> 008.png
  - iShot_2026-02-26_19.31.44.png -> 009.png
  - iShot_2026-02-26_19.31.51.png -> 010.png
  - iShot_2026-02-26_19.31.58.png -> 011.png
  - iShot_2026-02-26_19.32.05.png -> 012.png
  - iShot_2026-02-26_19.32.35.png -> 013.png
  - iShot_2026-02-26_19.32.41.png -> 014.png
  - iShot_2026-02-26_19.32.49.png -> 015.png
  - iShot_2026-02-26_19.32.54.png -> 016.png
  - iShot_2026-02-26_19.33.01.png -> 017.png
  - iShot_2026-02-26_19.33.08.png -> 018.png
  - iShot_2026-02-26_19.33.13.png -> 019.png
  - iShot_2026-02-26_19.33.24.png -> 020.png
  - iShot_2026-02-26_19.33.29.png -> 021.png
  - iShot_2026-02-26_19.33.37.png -> 022.pn

In [13]:
# 导入 OCR 模块
import sys
sys.path.insert(0, os.getcwd())
from ocr import OCR as TextInOCR, chat as chat_llm

In [14]:
def process_image_to_csv(img_path, output_dir):
    """
    处理单张图片：OCR识别 -> LLM纠错 -> 保存CSV
    
    参数:
        img_path: 图片文件路径
        output_dir: 输出目录
    
    返回:
        csv_path: 生成的CSV文件路径
    """
    # 获取文件名（不含扩展名）
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    csv_name = f"{img_name}.csv"
    csv_path = os.path.join(output_dir, csv_name)
    
    # 断点恢复：如果CSV已存在，则跳过
    if os.path.exists(csv_path):
        print(f"  [跳过] {img_name}.csv 已存在")
        return csv_path
    
    print(f"  [处理] {img_name}...")
    
    # Step 1: OCR 识别
    print(f"    -> OCR识别中...")
    ocr_response = TextInOCR(img_file=img_path)
    
    # 解析 OCR 结果，提取 markdown/表格内容
    try:
        ocr_result = json.loads(ocr_response)
        # 尝试提取 markdown 或 elements 中的文本
        if "result" in ocr_result:
            if "markdown" in ocr_result["result"]:
                ocr_text = ocr_result["result"]["markdown"]
            elif "elements" in ocr_result["result"] and ocr_result["result"]["elements"]:
                # 取第一个元素的文本
                ocr_text = ocr_result["result"]["elements"][0].get("text", "")
            else:
                ocr_text = str(ocr_result)
        else:
            ocr_text = str(ocr_result)
    except Exception as e:
        print(f"    -> OCR解析失败: {e}")
        ocr_text = str(ocr_response)
    
    # Step 2: LLM 纠错
    print(f"    -> LLM纠错中...")
    
    # 构建 prompt
    prompt = f"""请仔细校对以下从手写礼簿OCR识别出的文本，并纠正可能的错误。

OCR识别结果：
```
{ocr_text}
```
===
你是一名精通民俗礼簿记录的校对专家。
识别与校对规则：这可能是中文，如‘え’改为‘元’。特殊的字符进行校对
输出要求：转化为CSV格式。回答省略思考过程与评论，直接给出csv格式。字段为 姓名,金额(大写),金额(数字),地址(备注)。<code> 封装结果
"""
    
    llm_result = chat_llm(prompt)
    
    # 清理 LLM 输出
    csv_content = llm_result.replace('```', '').replace('csv', '').strip()
    
    # 添加表头
    # csv_with_header = "姓名,金额(大写),金额(数字),地址(备注)\n" + csv_content
    csv_with_header = csv_content
    
    # 保存CSV
    with open(csv_path, 'w', encoding='utf-8') as f:
        f.write(csv_with_header)
    
    print(f"    -> 已保存: {csv_name}")
    
    return csv_path

print("函数 process_image_to_csv 已定义完成")

函数 process_image_to_csv 已定义完成


In [15]:
# 批量处理所有图片
processed_files = []
skipped_files = []

print("=" * 50)
print("开始批量处理图片")
print("=" * 50)

for img_path in image_files:
    img_name = os.path.basename(img_path)
    
    try:
        csv_path = process_image_to_csv(img_path, OUTPUT_DIR)
        processed_files.append(csv_path)
    except Exception as e:
        print(f"  [错误] {img_name}/: {e}")
        skipped_files.append(img_name)

print("=" * 50)
print(f"处理完成！共处理 {len(processed_files)} 张图片")
if skipped_files:
    print(f"跳过 {len(skipped_files)} 张（已存在CSV）")

开始批量处理图片
  [处理] 001...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 001.csv
  [处理] 002...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 002.csv
  [处理] 003...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 003.csv
  [处理] 004...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 004.csv
  [处理] 005...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 005.csv
  [处理] 006...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 006.csv
  [处理] 007...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 007.csv
  [处理] 008...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 008.csv
  [处理] 009...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 009.csv
  [处理] 010...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 010.csv
  [处理] 011...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 011.csv
  [处理] 012...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 012.csv
  [处理] 013...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 013.csv
  [处理] 014...
    -> OCR识别中...
    -> LLM纠错中...
    -> 已保存: 014.csv
  [处理] 015...
    -> OCR识别中...
    -> L

In [16]:
# 查看生成的CSV文件
csv_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.csv")))
print(f"共生成 {len(csv_files)} 个CSV文件:\n")

for csv_file in csv_files:
    print(f"=== {os.path.basename(csv_file)} ===")
    df = pd.read_csv(csv_file)
    print(df.to_string(index=False))
    print()

共生成 55 个CSV文件:

=== 001.csv ===
 姓名 金额(大写)  金额(数字) 地址(备注)
周西昌    壹仟元    1000     罗沙
严文军    贰佰元     200     猴场
韦辅先    叁佰元     300    NaN
陈超进    叁佰元     300     下冲
陈九群    贰佰元     200      元
陈九久    贰佰元     200      元
 陈杰    贰佰元     200      元
梁德江    叁佰元     300     罗甸
汤允贵    贰佰元     200     车昆
 黄伟    伍佰元     500    NaN

=== 002.csv ===
 姓名 金额(大写)  金额(数字) 地址(备注)
王良周     叁佰     300   永龙玉石
 永兵     叁佰     300   贵阳解行
杨志立     贰佰     200    张家口
郭正龙     贰佰     200    NaN
相世林     贰佰     200    水井爹
李明志     贰佰     200    盘龙差
李兴发     贰佰     200    解兴路
王永龙     叁佰     300     张家
王永祥     贰佰     200     张家
张兴兵     贰佰     200    NaN

=== 003.csv ===
 姓名 金额(大写)  金额(数字) 地址(备注)
张兴红     伍佰     500     张家
张兴江     贰佰     200      、
汪艾祥     贰佰     200    信邦城
陈家发     叁佰     300     下冲
杨秀林     贰佰     200     兄布
杨洪林     贰佰     200      い
瘦仕武     贰佰     200     都還
 敖健     壹佰     100     盘龙
陈应付     叁佰     300      い
陈顺发      贰       2      い

=== 004.csv ===
 姓名 金额(大写)  金额(数字) 地址(备注)
胡仕国     贰仟    2000     床井
邹祖光    

ParserError: Error tokenizing data. C error: Expected 4 fields in line 23, saw 5


In [19]:
# 合并所有 CSV 文件
print("=" * 50)
print("开始合并所有CSV文件")
print("=" * 50)

all_dataframes = []

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        df["来源文件"] = os.path.basename(csv_file)
        all_dataframes.append(df)
    except Exception as e:
        print(f"读取失败 {csv_file}: {e}")

# 合并数据
if all_dataframes:
    merged_df = pd.concat(all_dataframes, ignore_index=True)
    
    # 查看合并后的数据
    print(f"\n合并后共 {len(merged_df)} 条记录")
    print("\n前10条记录：")
    print(merged_df.head(10).to_string(index=False))
    
    # 保存合并后的文件
    final_output = "礼簿_合并.csv"
    merged_df.to_csv(final_output, index=False, encoding='utf-8-sig')
    print(f"\n已保存合并结果: {final_output}")
else:
    print("没有可合并的数据")

开始合并所有CSV文件

合并后共 550 条记录

前10条记录：
 姓名 金额(大写) 金额(数字) 地址(备注)    来源文件
周西昌    壹仟元   1000     罗沙 001.csv
严文军    贰佰元    200     猴场 001.csv
韦辅先    叁佰元    300    NaN 001.csv
陈超进    叁佰元    300     下冲 001.csv
陈元辉    贰佰元    200     下冲 001.csv
陈元久    贰佰元    200     下冲 001.csv
 陈杰    贰佰元    200     下冲 001.csv
梁德江    叁佰元    300     罗甸 001.csv
汤元贵    贰佰元    200     本寨 001.csv
 黄伟    伍佰元    500    NaN 001.csv

已保存合并结果: 礼簿_合并.csv


# 生成html

In [19]:
# ,,0,
import pandas as pd
import cn2an, os
# 读取合并后的数据并添加姓氏字母和录入序号
if os.path.exists("礼簿_合并.csv"):
    final_df = pd.read_csv("礼簿_合并.csv")
    final_df["金额(数字)"] = final_df["金额(数字)"].astype(int)
    
    # ============================================================
    # 使用 cn2an 根据数字金额重新生成大写金额
    # ============================================================
    def convert_to_chinese_amount(amount):
        """将数字金额转换为中文大写金额（精确到个位数）"""
        try:
            if pd.isna(amount):
                return ""
            # 使用 cn2an 转换为中文大写，mode='simple' 表示简体中文
            chinese = cn2an.an2cn(amount, mode='rmb')
            # 删除"元"字
            return chinese[:-1].strip('元')+'圆'
        except Exception as e:
            print(f"转换失败: {amount}, 错误: {e}")
            return str(amount)
    
    # 重新生成"金额(大写)"列
    final_df["金额(大写)"] = final_df["金额(数字)"].apply(convert_to_chinese_amount)
    print("已使用 cn2an 重新生成大写金额")
    # ============================================================
    
    # 添加姓氏字母（取姓名第一个字的拼音首字母）
    # 这里需要安装拼音库，如果没有可以手动添加
    try:
        import pypinyin
        def get_surname_letter(name):
            if pd.isna(name) or len(str(name)) == 0:
                return ""
            first_char = str(name)[0]
            pinyin = pypinyin.lazy_pinyin(first_char)[0][0]
            return pinyin[0].upper()
        final_df.insert(0, "姓氏字母", final_df["姓名"].apply(get_surname_letter))
    except ImportError:
        print("提示：未安装 pypinyin 库，姓氏字母列将留空")
        final_df.insert(0, "姓氏字母", "")
    
    # 添加录入序号（可选）
    final_df["录入序号"] = range(1, len(final_df) + 1)
    
    # 调整列顺序
    column_order = ["姓氏字母", "姓名", "金额(大写)", "金额(数字)", "地址(备注)", "录入序号"]
    final_df = final_df[column_order]
    
    # 保存最终结果
    final_df.to_csv("礼簿_最终版.csv", index=False, encoding='utf-8-sig')
    
    print("最终礼簿数据：")
    print(final_df.head(20).to_string(index=False))
    print(f"\n共 {len(final_df)} 条记录")
    print("\n已保存至: 礼簿_最终版.csv")
else:
    print("请先执行上一步生成合并CSV")

已使用 cn2an 重新生成大写金额
最终礼簿数据：
姓氏字母   姓名 金额(大写)  金额(数字) 地址(备注)  录入序号
   Y  叶德平    壹仟圆    1000     罗沙     1
   S   石刚    贰佰圆     200     猴场     2
   M  孟建华    叁佰圆     300    NaN     3
   L  梁小梅    叁佰圆     300     下冲     4
   O 欧阳金英    贰佰圆     200     下冲     5
   K  康秀兰    贰佰圆     200     下冲     6
   C   曾金    贰佰圆     200     下冲     7
   L   林桂    叁佰圆     300     罗甸     8
   X  谢永伟    贰佰圆     200     本寨     9
   Y   余财    伍佰圆     500    NaN    10
   Q   乔强    贰佰圆     200   永龙玉石    11
   S   宋义    叁佰圆     300   贵阳银行    12
   Q  邱秀华    贰佰圆     200    张家口    13
   W  魏建国    贰佰圆     200    NaN    14
   L   廖忠    贰佰圆     200    水井弯    15
   S  苏红珍    贰佰圆     200    盘龙巷    16
   Y  袁永伟    贰佰圆     200    解兴路    17
   W   王燕    叁佰圆     300    张家口    18
   L  刘德平    贰佰圆     200    张家口    19
   D   董敏    贰佰圆     200    张家口    20

共 551 条记录

已保存至: 礼簿_最终版.csv


In [20]:
import unicodedata

def collect_suspicious_chars(series, desc=""):
    freq = {}
    for s in series.dropna():
        for ch in str(s):
            # 放过常用汉字区 + 常见符号和空格
            if "\u4e00" <= ch <= "\u9fff":
                continue
            if ch in " 　·.-_，,./()（）0123456789":
                continue
            freq[ch] = freq.get(ch, 0) + 1

    if not freq:
        print(f"{desc} 未发现可疑字符")
        return

    print(f"{desc} 中发现的可疑字符：")
    for ch, cnt in sorted(freq.items(), key=lambda x: -x[1]):
        code = f"U+{ord(ch):04X}"
        name = unicodedata.name(ch, "UNKNOWN")
        print(f"{repr(ch)}  次数={cnt}  编码={code}  名称={name}")

# 在姓名字段里找一找
collect_suspicious_chars(final_df["姓名"], desc="姓名列")

姓名列 未发现可疑字符


In [21]:
import pandas as pd
final_df = pd.read_csv("礼簿_最终版.csv").fillna("")
# 去除空格
final_df = final_df.applymap(lambda x: str(x).replace(" ", "") if isinstance(x, str) else x)
for index,row in final_df.iterrows():
    print(f'{{ capital:"{row["姓氏字母"]}", name: "{row["姓名"]}", words: "{row["金额(大写)"]}", num: {row["金额(数字)"]}, addr: "{row.get("地址(备注)", "")}", id: {row["录入序号"]} }},')

{ capital:"Y", name: "叶德平", words: "壹仟圆", num: 1000, addr: "罗沙", id: 1 },
{ capital:"S", name: "石刚", words: "贰佰圆", num: 200, addr: "猴场", id: 2 },
{ capital:"M", name: "孟建华", words: "叁佰圆", num: 300, addr: "", id: 3 },
{ capital:"L", name: "梁小梅", words: "叁佰圆", num: 300, addr: "下冲", id: 4 },
{ capital:"O", name: "欧阳金英", words: "贰佰圆", num: 200, addr: "下冲", id: 5 },
{ capital:"K", name: "康秀兰", words: "贰佰圆", num: 200, addr: "下冲", id: 6 },
{ capital:"C", name: "曾金", words: "贰佰圆", num: 200, addr: "下冲", id: 7 },
{ capital:"L", name: "林桂", words: "叁佰圆", num: 300, addr: "罗甸", id: 8 },
{ capital:"X", name: "谢永伟", words: "贰佰圆", num: 200, addr: "本寨", id: 9 },
{ capital:"Y", name: "余财", words: "伍佰圆", num: 500, addr: "", id: 10 },
{ capital:"Q", name: "乔强", words: "贰佰圆", num: 200, addr: "永龙玉石", id: 11 },
{ capital:"S", name: "宋义", words: "叁佰圆", num: 300, addr: "贵阳银行", id: 12 },
{ capital:"Q", name: "邱秀华", words: "贰佰圆", num: 200, addr: "张家口", id: 13 },
{ capital:"W", name: "魏建国", words: "贰佰圆", num: 200

/var/folders/2z/pt5yj8rx2lvgfb60vfwvppd00000gn/T/ipykernel_92983/1576427173.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_df = final_df.applymap(lambda x: str(x).replace(" ", "") if isinstance(x, str) else x)


In [22]:
# final_df[final_df['姓名'].duplicated()].sort_values(by="姓名").to_csv('xxx.csv')

In [23]:
# 生成 HTML 礼簿（可自定义标题）- 读取外部模板
def generate_html(final_df, output_path="礼簿_网页版.html", 
                  title="周儒福&郭向丽结婚礼簿", 
                  date="2026年2月11日",
                  blessing="百年好合 · 永结同心",
                  template_path="red_template.html"):
    """
    根据 final_df 生成完整的 HTML 礼簿
    
    参数:
        final_df: 包含礼簿数据的 DataFrame
        output_path: 输出 HTML 文件路径
        title: 礼簿标题（如：XXX结婚礼簿）
        date: 礼簿日期
        blessing: 祝福语
        template_path: HTML 模板文件路径
    """
    # 生成 rawData JavaScript 代码
    raw_data_lines = []
    for index, row in final_df.iterrows():
        name = row["姓名"]
        capital = row["姓氏字母"]
        words = row["金额(大写)"]
        num = int(row["金额(数字)"])
        if num == 0:
            # num = ""
            words = ""
        addr = row.get("地址(备注)", "")
        if pd.isna(addr):
            addr = ""
        row_id = int(row["录入序号"])
        
        raw_data_lines.append(f'{{ capital:"{capital}", name: "{name}", words: "{words}", num: {num}, addr: "{addr}", id: {row_id} }}')
    
    raw_data_str = ",\n        ".join(raw_data_lines)
    
    # 读取 HTML 模板
    with open(template_path, 'r', encoding='utf-8') as f:
        html_template = f.read()
    
    # 替换模板中的占位符
    html_template = html_template.replace('{{TITLE}}', title)
    html_template = html_template.replace('{{DATE}}', date)
    html_template = html_template.replace('{{BLESSING}}', blessing)
    html_template = html_template.replace('{{RAWDATA}}', raw_data_str)
    
    # 写入文件
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_template)
    
    print(f"✅ HTML 礼簿已生成: {output_path}")
    print(f"   标题: {title}")
    print(f"   日期: {date}")
    print(f"   祝福语: {blessing}")
    print(f"   记录数: {len(final_df[final_df['金额(数字)']>0])}")

print("函数 generate_html 已定义完成")

函数 generate_html 已定义完成


## 生成原版html

In [24]:
# 生成 HTML 礼簿
# 根据你的礼簿修改以下参数：
generate_html(
    final_df, 
    output_path="原版_陈元和出嫁女儿礼簿.html",
    title="原版_陈元和出嫁女儿礼簿",    # 标题
    date="2026年2月28日",             # 日期
    blessing="大壮&小美\n百年好合 · 永结同心"      # 祝福语
)

✅ HTML 礼簿已生成: 原版_陈元和出嫁女儿礼簿.html
   标题: 原版_陈元和出嫁女儿礼簿
   日期: 2026年2月28日
   祝福语: 大壮&小美
百年好合 · 永结同心
   记录数: 527


In [25]:
final_df['地址(备注)'].value_counts()

地址(备注)
        126
下冲       44
羊场       25
土坝       12
大塘       12
       ... 
紫郡        1
栗木小学      1
尖坡        1
桐树        1
联社        1
Name: count, Length: 153, dtype: int64

In [26]:
from pypinyin import lazy_pinyin, Style
import pandas as pd

def split_name_and_pinyin(name):
    s = str(name)
    if not s:
        return pd.Series(["", "", "", ""])
    
    # 姓和名
    surname = s[0]
    given_name = s[1:]
    
    # 姓的拼音
    surname_py = lazy_pinyin(surname, style=Style.NORMAL)[0]
    
    # 名的拼音（每个字单独，再用分隔符连接）
    if given_name:
        given_py_list = lazy_pinyin(given_name, style=Style.NORMAL)
        given_py = "-".join(given_py_list)   # 这里可以换成你想要的分隔符
    else:
        given_py = ""
    
    return pd.Series([surname, given_name, surname_py, given_py])

# 生成辅助列
final_df[["姓氏", "名字", "姓氏拼音", "名字拼音"]] = final_df["姓名"].apply(split_name_and_pinyin)

# 多级排序：先按姓氏拼音，再按姓氏汉字本身，最后按名字拼音
sorted_df = final_df.sort_values(
    by=["姓氏拼音", "姓氏", "名字拼音"],
    ascending=[True, True, True]
)

sorted_df

,姓氏字母,姓名,金额(大写),金额(数字),地址(备注),录入序号,姓氏,名字,姓氏拼音,名字拼音
117,,,零圆,0,,118,,,,
207,,,零圆,0,,208,,,,
426,,,零圆,0,,427,,,,
427,,,零圆,0,,428,,,,
428,,,零圆,0,,429,,,,
...,...,...,...,...,...,...,...,...,...,...
424,Z,朱玉梅,贰佰圆,200,下冲,425,朱,玉梅,zhu,yu-mei
294,Z,邹美兰,贰佰圆,200,,295,邹,美兰,zou,mei-lan
27,Z,邹涛,壹佰圆,100,盘龙巷,28,邹,涛,zou,tao
256,Z,邹文忠,贰佰圆,200,新农贸,257,邹,文忠,zou,wen-zhong


## 生成排序版html

In [27]:
# 生成 HTML 礼簿
# 根据你的礼簿修改以下参数：
generate_html(
    sorted_df[sorted_df["金额(数字)"]>0],
    output_path="排序版_郭向丽结婚礼簿.html",
    title="排序版_郭向丽结婚礼簿",    # 标题
    date="2026年2月28日",             # 日期
    blessing="大壮&小美\n百年好合 · 永结同心"      # 祝福语
)

✅ HTML 礼簿已生成: 排序版_郭向丽结婚礼簿.html
   标题: 排序版_郭向丽结婚礼簿
   日期: 2026年2月28日
   祝福语: 大壮&小美
百年好合 · 永结同心
   记录数: 527


In [28]:
# generate_html(
#     final_df, 
#     output_path="白事礼簿_网页版.html",
#     title="治丧礼簿",    # 标题
#     date="2026年2月11日",             # 日期
#     blessing="音容宛在 · 德范长存",      # 祝福语
#     template_path="black_template.html"
# )